# Тест 2 +

## Задание 1

Из БВ FRED загрузите ряд `BAA` ([ссылка](https://fred.stlouisfed.org/series/BAA)) c **2000-01-01** по **2025-12-31**.Пусть `y` - первая разность это ряда. Для ряда `y` выберите оптимальный порядок модели AR(L)-GARCH (p,o,q) с распределением для нормализованных шоков из списка: **['normal', 't', 'skewt']** Используйте метод кросс-валидации: **расширяющаяся** обучающая выборка. Используйте следующие параметры кросс-валидации: длина (первоначальной) обучающей выборки - 100, горизонт прогнозирования - 5 периодов, сдвиг - 5 периодов. Метрика кросс-валидации **MAE**. Рассмотрите следующие диапазоны: 0≤L≤3, 1≤p≤2, 0≤o≤2, 1≤q≤2. Остальные параметры модели по умолчанию. 

**Указание**: не забудьте задать правильный периодический индекс для ряда `y`

L=Ответ {$а} Вопрос 1

p=Ответ {$а} Вопрос 1

o=Ответ {$а} Вопрос 1

q=Ответ {$а} Вопрос 1

Распределение шоков: Ответ {$а} Вопрос 1 normaltskewt

In [1]:
import warnings
warnings.filterwarnings("ignore")

import itertools
import numpy as np
import pandas as pd
from pandas_datareader.data import DataReader
from arch import arch_model

x = DataReader(
    "BAA",
    "fred",
    start="2000-01-01",
    end="2025-12-31",
)["BAA"].dropna()

# BAA - месячный ряд
x.index = x.index.to_period("M")

y = x.diff().dropna()

initial_train = 100
horizon = 5
step = 5

Ls = range(0, 4)
ps = range(1, 3)
os = range(0, 3)
qs = range(1, 3)
dists = ["normal", "t", "skewt"]

results = []

for L, p, o, q, dist in itertools.product(Ls, ps, os, qs, dists):
    abs_errors = []

    for train_end in range(initial_train, len(y) - horizon + 1, step):
        y_train = y.iloc[:train_end]
        y_test = y.iloc[train_end:train_end + horizon]

        model = arch_model(
            y_train,
            mean="ARX",
            lags=L,
            vol="GARCH",
            p=p,
            o=o,
            q=q,
            dist=dist,
        )

        fit = model.fit(disp="off")
        pred = fit.forecast(horizon=horizon, reindex=False).mean.iloc[-1]

        abs_errors.extend(np.abs(y_test.to_numpy() - pred.to_numpy()))

    results.append({
        "L": L,
        "p": p,
        "o": o,
        "q": q,
        "dist": dist,
        "mae": np.mean(abs_errors),
    })

cv_results = pd.DataFrame(results).sort_values("mae")
print(cv_results.head(10))

best = cv_results.iloc[0]
print(best)

    L  p  o  q    dist       mae
64  1  2  1  2       t  0.154494
51  1  1  2  2  normal  0.154512
48  1  1  2  1  normal  0.154513
66  1  2  2  1  normal  0.154539
69  1  2  2  2  normal  0.154539
46  1  1  1  2       t  0.154558
81  2  1  1  2  normal  0.154562
78  2  1  1  1  normal  0.154562
43  1  1  1  1       t  0.154564
99  2  2  1  2  normal  0.154582
L              1
p              2
o              1
q              2
dist           t
mae     0.154494
Name: 64, dtype: object
